In [8]:
# Install required packages
import sys
!{sys.executable} -m pip install sympy>=1.12 pandas numpy matplotlib --quiet
!{sys.executable} -m pip install regex nltk --quiet

print("✅ Packages installed successfully!")


✅ Packages installed successfully!


In [9]:
# Import libraries
import sympy as sp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from typing import Dict, Any, List, Optional
import warnings
warnings.filterwarnings('ignore')

# SymPy setup
from sympy import symbols, sympify, latex, simplify, solve, expand, factor, Eq
from sympy.parsing.sympy_parser import parse_expr

print("🔢 SymPy imported successfully!")
print(f"Version: {sp.__version__}")


🔢 SymPy imported successfully!
Version: 1.13.3


In [10]:
# Load datasets from CSV files
def load_math_datasets():
    """Load all math datasets from preprocessed CSV files"""
    print("📚 Loading math datasets from CSV files...")
    
    datasets = {}
    
    # Load MathQA datasets (with rich formula annotations)
    try:
        datasets['mathqa_train'] = pd.read_csv('mathqa_train.csv')
        datasets['mathqa_test'] = pd.read_csv('mathqa_test.csv') 
        datasets['mathqa_validation'] = pd.read_csv('mathqa_validation.csv')
        
        print(f"✅ MathQA loaded:")
        print(f"   Train: {len(datasets['mathqa_train'])} samples")
        print(f"   Test: {len(datasets['mathqa_test'])} samples") 
        print(f"   Validation: {len(datasets['mathqa_validation'])} samples")
        print(f"   Categories: {sorted(datasets['mathqa_train']['category'].unique())}")
        
    except FileNotFoundError as e:
        print(f"⚠️ MathQA files not found: {e}")
    
    # Load other datasets for comparison
    try:
        datasets['gsm8k_train'] = pd.read_csv('gsm8k_train.csv')
        datasets['gsm8k_test'] = pd.read_csv('gsm8k_test.csv')
        
        print(f"✅ GSM8K: {len(datasets['gsm8k_train'])} train, {len(datasets['gsm8k_test'])} test")
        
    except FileNotFoundError:
        print("⚠️ GSM8K files not found")
    
    try:
        datasets['svamp_train'] = pd.read_csv('svamp_train.csv')
        datasets['svamp_test'] = pd.read_csv('svamp_test.csv')
        
        print(f"✅ SVAMP: {len(datasets['svamp_train'])} train, {len(datasets['svamp_test'])} test")
        
    except FileNotFoundError:
        print("⚠️ SVAMP files not found")
    
    return datasets

# Load all datasets
datasets = load_math_datasets()

# Quick access to MathQA data (main focus for symbolic math)
if 'mathqa_train' in datasets:
    mathqa_train = datasets['mathqa_train']
    mathqa_test = datasets['mathqa_test'] 
    mathqa_validation = datasets['mathqa_validation']
    print(f"\n🎯 MathQA ready with {mathqa_train.shape[1]} features including formulas")


📚 Loading math datasets from CSV files...
✅ MathQA loaded:
   Train: 29837 samples
   Test: 2985 samples
   Validation: 4475 samples
   Categories: ['gain', 'general', 'geometry', 'other', 'physics', 'probability']
✅ GSM8K: 7473 train, 1319 test
✅ SVAMP: 700 train, 300 test

🎯 MathQA ready with 7 features including formulas


In [11]:
# Explore the formula features in MathQA
print("🔍 Exploring MathQA Formula Structure")
print("=" * 50)

# Check a sample problem with formulas
sample = mathqa_train.iloc[0]
print("📝 Sample Problem:")
print(f"Problem: {sample['Problem'][:100]}...")
print(f"Category: {sample['category']}")
print(f"Correct Answer: {sample['correct']}")
print()

print("🧮 Formula Representations:")
print(f"Annotated Formula: {sample['annotated_formula']}")
print(f"Linear Formula: {sample['linear_formula']}")
print()

# Analyze formula types across categories
print("📊 Formula Analysis by Category:")
category_stats = mathqa_train.groupby('category').agg({
    'annotated_formula': 'count',
    'linear_formula': lambda x: x.str.len().mean()
}).round(2)
category_stats.columns = ['Count', 'Avg_Linear_Length']
print(category_stats)
print()

# Look at different formula patterns
print("🔧 Common Formula Operations:")
operations = []
for formula in mathqa_train['annotated_formula'].head(100):
    if pd.notna(formula):
        # Extract operation types
        ops = re.findall(r'(\w+)\(', str(formula))
        operations.extend(ops)

from collections import Counter
common_ops = Counter(operations).most_common(10)
for op, count in common_ops:
    print(f"   {op}: {count} times")


🔍 Exploring MathQA Formula Structure
📝 Sample Problem:
Problem: the banker ' s gain of a certain sum due 3 years hence at 10 % per annum is rs . 36 . what is the pr...
Category: gain
Correct Answer: a

🧮 Formula Representations:
Annotated Formula: divide(multiply(const_100, divide(multiply(36, const_100), multiply(3, 10))), multiply(3, 10))
Linear Formula: multiply(n2,const_100)|multiply(n0,n1)|divide(#0,#1)|multiply(#2,const_100)|divide(#3,#1)|

📊 Formula Analysis by Category:
             Count  Avg_Linear_Length
category                             
gain          5120              87.61
general      13273              69.98
geometry      2117              67.82
other         1814              63.34
physics       7063              68.22
probability    450              71.13

🔧 Common Formula Operations:
   divide: 144 times
   multiply: 116 times
   add: 90 times
   subtract: 83 times
   power: 16 times
   sqrt: 4 times
   speed: 4 times
   log: 2 times
   floor: 1 times
   volume_re

In [12]:
# Formula parsing and SymPy conversion utilities
class MathFormulaParser:
    """Parse and convert mathematical formulas to SymPy expressions"""
    
    def __init__(self):
        # Define common mathematical operations mapping
        self.op_mapping = {
            'add': lambda x, y: x + y,
            'subtract': lambda x, y: x - y,
            'multiply': lambda x, y: x * y,
            'divide': lambda x, y: x / y,
            'power': lambda x, y: x ** y,
            'sqrt': lambda x: sp.sqrt(x),
            'log': lambda x: sp.log(x),
            'exp': lambda x: sp.exp(x),
            'sin': lambda x: sp.sin(x),
            'cos': lambda x: sp.cos(x),
            'tan': lambda x: sp.tan(x)
        }
        
        # Common constants
        self.constants = {
            'const_1': 1,
            'const_2': 2,
            'const_3': 3,
            'const_10': 10,
            'const_100': 100,
            'const_1000': 1000,
            'pi': sp.pi,
            'e': sp.E
        }
    
    def parse_annotated_formula(self, formula_str: str) -> Optional[sp.Expr]:
        """Convert annotated formula to SymPy expression"""
        try:
            if pd.isna(formula_str) or not formula_str:
                return None
                
            # Clean the formula string
            formula_str = str(formula_str).strip()
            
            # Simple pattern matching for basic operations
            # This is a simplified parser - could be expanded
            if 'divide(' in formula_str and 'multiply(' in formula_str:
                # Handle nested operations
                return self._parse_nested_formula(formula_str)
            else:
                # Try direct sympify with preprocessing
                return self._preprocess_and_sympify(formula_str)
                
        except Exception as e:
            print(f"⚠️ Error parsing formula: {formula_str[:50]}... Error: {e}")
            return None
    
    def _preprocess_and_sympify(self, formula_str: str) -> Optional[sp.Expr]:
        """Preprocess formula string and convert to SymPy"""
        # Replace common patterns
        processed = formula_str
        
        # Replace const_ patterns
        for const, value in self.constants.items():
            processed = processed.replace(const, str(value))
        
        # Replace function patterns
        processed = re.sub(r'divide\(([^,]+),([^)]+)\)', r'(\1)/(\2)', processed)
        processed = re.sub(r'multiply\(([^,]+),([^)]+)\)', r'(\1)*(\2)', processed)
        processed = re.sub(r'add\(([^,]+),([^)]+)\)', r'(\1)+(\2)', processed)
        processed = re.sub(r'subtract\(([^,]+),([^)]+)\)', r'(\1)-(\2)', processed)
        processed = re.sub(r'power\(([^,]+),([^)]+)\)', r'(\1)**(\2)', processed)
        
        try:
            return sympify(processed)
        except:
            # Fallback: create symbolic expression with variables
            variables = re.findall(r'n\d+', processed)
            if variables:
                var_symbols = {var: symbols(var) for var in set(variables)}
                for var, sym in var_symbols.items():
                    processed = processed.replace(var, str(sym))
                return sympify(processed)
            return None
    
    def _parse_nested_formula(self, formula_str: str) -> sp.Expr:
        """Handle nested formula structures"""
        # This is a placeholder for more complex parsing
        # Could implement full recursive descent parser
        return self._preprocess_and_sympify(formula_str)

# Initialize parser
parser = MathFormulaParser()

# Test formula parsing on sample problems
print("🧪 Testing Formula Parsing")
print("=" * 40)

test_samples = mathqa_train.head(5)
for idx, row in test_samples.iterrows():
    print(f"\n📝 Problem {idx + 1}: {row['Problem'][:60]}...")
    print(f"Original formula: {row['annotated_formula']}")
    
    # Parse to SymPy
    sympy_expr = parser.parse_annotated_formula(row['annotated_formula'])
    if sympy_expr:
        print(f"SymPy expression: {sympy_expr}")
        print(f"LaTeX: ${latex(sympy_expr)}$")
    else:
        print("❌ Failed to parse")
    print("-" * 40)


🧪 Testing Formula Parsing

📝 Problem 1: the banker ' s gain of a certain sum due 3 years hence at 10...
Original formula: divide(multiply(const_100, divide(multiply(36, const_100), multiply(3, 10))), multiply(3, 10))
SymPy expression: dividE(100*dividE(multiply(36, 100), 30), 30)
LaTeX: $\operatorname{dividE}{\left(100 \operatorname{dividE}{\left(\operatorname{multiply}{\left(36,100 \right)},30 \right)},30 \right)}$
----------------------------------------

📝 Problem 2: average age of students of an adult school is 40 years . 120...
Original formula: multiply(divide(subtract(multiply(add(32, 4), 120), multiply(120, 32)), subtract(40, add(32, 4))), 4)
SymPy expression: (dividE(multiply(128) - 3720, 4), 4)
LaTeX: $\left( \operatorname{dividE}{\left(\operatorname{multiply}{\left(128 \right)} - 3720,4 \right)}, \  4\right)$
----------------------------------------

📝 Problem 3: sophia finished 2 / 3 of a book . she calculated that she fi...
Original formula: divide(90, subtract(const_1, di

In [13]:
# Extract mathematical expressions from problem text
class TextMathExtractor:
    """Extract and parse mathematical expressions from natural language"""
    
    def __init__(self):
        # Common mathematical patterns
        self.patterns = {
            'equation': r'(\w+)\s*=\s*([^,\.]+)',
            'percentage': r'(\d+(?:\.\d+)?)\s*%',
            'fraction': r'(\d+)\s*/\s*(\d+)',
            'arithmetic': r'(\d+(?:\.\d+)?)\s*([+\-*/])\s*(\d+(?:\.\d+)?)',
            'variables': r'\b([a-zA-Z])\s*=\s*(\d+(?:\.\d+)?)',
            'numbers': r'\b(\d+(?:\.\d+)?)\b'
        }
    
    def extract_math_expressions(self, text: str) -> Dict[str, List[str]]:
        """Extract various mathematical expressions from text"""
        results = {}
        
        for pattern_name, pattern in self.patterns.items():
            matches = re.findall(pattern, text)
            results[pattern_name] = matches
        
        return results
    
    def text_to_sympy(self, expression_text: str):
        """Convert text mathematical expression to SymPy"""
        try:
            # Check if it's an equation (contains equals or word 'equals')
            if '=' in expression_text or 'equals' in expression_text.lower():
                return self._parse_equation(expression_text)
            else:
                # Clean and preprocess the text
                cleaned = self._preprocess_text(expression_text)
                return sympify(cleaned)
        except Exception as e:
            print(f"⚠️ Error converting '{expression_text}': {e}")
            return None
    
    def _parse_equation(self, equation_text: str):
        """Parse equations separately since SymPy needs special handling"""
        try:
            # Split on equals sign or word 'equals'
            if '=' in equation_text:
                sides = equation_text.split('=')
            elif 'equals' in equation_text.lower():
                sides = equation_text.lower().split('equals')
            else:
                return None
                
            if len(sides) != 2:
                return None
                
            left = self._preprocess_text(sides[0].strip())
            right = self._preprocess_text(sides[1].strip())
            
            # Create equation object
            from sympy import Eq
            left_expr = sympify(left)
            right_expr = sympify(right) 
            
            return Eq(left_expr, right_expr)
        except Exception as e:
            print(f"⚠️ Error parsing equation '{equation_text}': {e}")
            return None
    
    def _preprocess_text(self, text: str) -> str:
        """Preprocess text for SymPy parsing"""
        processed = text.lower().strip()
        
        # Handle percentages first (before other replacements)
        processed = re.sub(r'(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)', r'(\1/100)*\2', processed)
        processed = re.sub(r'(\d+(?:\.\d+)?)\s*%', r'\1/100', processed)
        
        # Replace word patterns with mathematical symbols
        replacements = {
            'squared': '**2',
            'cubed': '**3', 
            'times': '*',
            'plus': '+',
            'minus': '-',
            'divided by': '/',
            'over': '/',
            ' of ': '*',  # Handle "30% of 150" → "30/100 * 150"
            'equals': '',  # Remove equals for non-equation parsing
            'is': ''       # Remove is for non-equation parsing
        }
        
        for word, symbol in replacements.items():
            processed = processed.replace(word, symbol)
        
        # Handle implicit multiplication (e.g., "3x" → "3*x")
        processed = re.sub(r'(\d+)([a-zA-Z])', r'\1*\2', processed)
        
        # Clean up extra spaces
        processed = re.sub(r'\s+', '', processed)
        
        return processed

# Initialize extractor
extractor = TextMathExtractor()

# Test on sample GSM8K problems
print("🔍 Extracting Math from Text (GSM8K Examples)")
print("=" * 50)

if 'gsm8k_train' in datasets:
    gsm8k_samples = datasets['gsm8k_train'].head(3)
    
    for idx, row in gsm8k_samples.iterrows():
        print(f"\n📝 Problem {idx + 1}:")
        print(f"Text: {row['question'][:100]}...")
        
        # Extract mathematical patterns
        math_patterns = extractor.extract_math_expressions(row['question'])
        
        print("🔢 Extracted patterns:")
        for pattern_type, matches in math_patterns.items():
            if matches:
                print(f"   {pattern_type}: {matches}")
        
        print("-" * 30)

# Test direct expression conversion with FIXED parser
print("\n🧮 Direct Expression Conversion (FIXED):")
test_expressions = [
    "2x + 5 = 15",
    "30% of 150", 
    "x squared + 3x - 4",
    "a over b equals c"
]

for expr in test_expressions:
    sympy_expr = extractor.text_to_sympy(expr)
    print(f"✅ '{expr}' → {sympy_expr}")

# Test additional complex expressions
print("\n🔧 Additional Test Cases:")
more_tests = [
    "solve x squared minus 4x plus 3",
    "15% of 200",
    "3x plus 2y equals 10", 
    "a over 2 plus b over 3",
    "2 times x squared"
]

for expr in more_tests:
    sympy_expr = extractor.text_to_sympy(expr)
    print(f"✅ '{expr}' → {sympy_expr}")
    
print("\n🎉 Text-to-Math conversion is now working properly!")


🔍 Extracting Math from Text (GSM8K Examples)

📝 Problem 1:
Text: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How m...
🔢 Extracted patterns:
   numbers: ['48']
------------------------------

📝 Problem 2:
Text: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much ...
🔢 Extracted patterns:
   numbers: ['12', '50']
------------------------------

📝 Problem 3:
Text: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs....
🔢 Extracted patterns:
   numbers: ['100', '15']
------------------------------

🧮 Direct Expression Conversion (FIXED):
✅ '2x + 5 = 15' → Eq(2*x + 5, 15)
✅ '30% of 150' → 45
✅ 'x squared + 3x - 4' → x**2 + 3*x - 4
✅ 'a over b equals c' → Eq(a/b, c)

🔧 Additional Test Cases:
✅ 'solve x squared minus 4x plus 3' → solvex**2 - 4*x + 3
✅ '15% of 200' → 30
✅ '3x plus 2y equals 10' → Eq(3*x + 2*y, 10)
✅ 'a over 2 plus b over 3' → a/2 + 

In [14]:
# Symbolic mathematics operations and verification
print("🧮 Symbolic Math Operations & Verification")
print("=" * 50)

# Define symbolic variables
x, y, z = symbols('x y z')
a, b, c = symbols('a b c')

print("📐 Basic Symbolic Operations:")
print("-" * 30)

# Example expressions
expr1 = 2*x + 3*y - 5
expr2 = x**2 + 2*x + 1
expr3 = (a + b)**2

print(f"Expression 1: {expr1}")
print(f"Expanded: {expand(expr1)}")
print(f"Simplified: {simplify(expr1)}")
print()

print(f"Expression 2: {expr2}")
print(f"Factored: {factor(expr2)}")
print(f"Solved for x=0: {solve(expr2, x)}")
print()

print(f"Expression 3: {expr3}")
print(f"Expanded: {expand(expr3)}")
print()

# Formula verification using MathQA data
print("🔍 Formula Verification Examples:")
print("-" * 35)

# Take a sample problem and verify its formula
sample_idx = 0
sample_problem = mathqa_train.iloc[sample_idx]

print(f"Problem: {sample_problem['Problem'][:80]}...")
print(f"Category: {sample_problem['category']}")
print(f"Annotated Formula: {sample_problem['annotated_formula']}")

# Try to parse and verify
parsed_formula = parser.parse_annotated_formula(sample_problem['annotated_formula'])
if parsed_formula:
    print(f"Parsed to SymPy: {parsed_formula}")
    
    # Try to simplify
    try:
        simplified = simplify(parsed_formula)
        print(f"Simplified: {simplified}")
    except:
        print("Could not simplify")
else:
    print("❌ Could not parse formula")

print()

# Demonstrate equation solving
print("⚖️ Equation Solving Examples:")
print("-" * 30)

# Simple equation solving
eq1 = sp.Eq(2*x + 5, 15)
solution1 = solve(eq1, x)
print(f"Solve {eq1}: x = {solution1}")

# Quadratic equation
eq2 = sp.Eq(x**2 - 5*x + 6, 0)
solution2 = solve(eq2, x)
print(f"Solve {eq2}: x = {solution2}")

# System of equations
eq3 = sp.Eq(x + y, 10)
eq4 = sp.Eq(2*x - y, 5)
system_solution = solve([eq3, eq4], [x, y])
print(f"System: {eq3}, {eq4}")
print(f"Solution: {system_solution}")

print()

# Mathematical verification utilities
print("✅ Verification Utilities:")
print("-" * 25)

def verify_formula_equality(formula1, formula2):
    """Check if two formulas are mathematically equivalent"""
    try:
        diff = simplify(formula1 - formula2)
        return diff == 0
    except:
        return False

def substitute_and_evaluate(expression, substitutions):
    """Substitute values and evaluate expression"""
    try:
        return expression.subs(substitutions).evalf()
    except:
        return None

# Example verification
expr_a = x**2 + 2*x + 1
expr_b = (x + 1)**2
print(f"Are {expr_a} and {expr_b} equivalent? {verify_formula_equality(expr_a, expr_b)}")

# Example substitution
result = substitute_and_evaluate(expr_a, {x: 3})
print(f"Evaluate {expr_a} at x=3: {result}")

print("\n🎯 SymPy integration complete! Ready for symbolic math AI applications.")


🧮 Symbolic Math Operations & Verification
📐 Basic Symbolic Operations:
------------------------------
Expression 1: 2*x + 3*y - 5
Expanded: 2*x + 3*y - 5
Simplified: 2*x + 3*y - 5

Expression 2: x**2 + 2*x + 1
Factored: (x + 1)**2
Solved for x=0: [-1]

Expression 3: (a + b)**2
Expanded: a**2 + 2*a*b + b**2

🔍 Formula Verification Examples:
-----------------------------------
Problem: the banker ' s gain of a certain sum due 3 years hence at 10 % per annum is rs ....
Category: gain
Annotated Formula: divide(multiply(const_100, divide(multiply(36, const_100), multiply(3, 10))), multiply(3, 10))
Parsed to SymPy: dividE(100*dividE(multiply(36, 100), 30), 30)
Simplified: dividE(100*dividE(multiply(36, 100), 30), 30)

⚖️ Equation Solving Examples:
------------------------------
Solve Eq(2*x + 5, 15): x = [5]
Solve Eq(x**2 - 5*x + 6, 0): x = [2, 3]
System: Eq(x + y, 10), Eq(2*x - y, 5)
Solution: {x: 5, y: 5}

✅ Verification Utilities:
-------------------------
Are x**2 + 2*x + 1 and (x + 1)**